
# Clean Experiment: Published Three-Stage Framework vs OAENet

This notebook rewrites the published synthetic experiment in a modular form and directly compares:

- **Enh-ELRT**: exposure logistic regression → tanh penalty smoothing → outcome elastic net → adaptive elastic net.
- **Enh-ESVMS**: exposure linear SVM → sigmoid penalty smoothing → outcome elastic net → adaptive elastic net.
- **OAENet**: outcome OLS → outcome-adaptive weights → penalized logistic exposure model.
- **Target**: oracle set \(X_C \cup X_P=\{X_1,X_2,X_3,X_4\}\).

The main published synthetic design uses \(p=20\), \(N\in\{200,500,1000\}\), \(\rho\in\{0,.25,.5,.75\}\), four scenarios, and seeds 1–30.

**Important source/implementation note.** The treatment DGP as typeset in the paper uses a Normal `epsilon_treat` threshold around `expit(X beta)`, which is unusual. This notebook defaults to the standard Bernoulli-logit interpretation, but also provides `treatment_mode="paper_literal"` so the printed equation can be run literally.

The papers separately parameterize L1 and L2 penalties. The exact mixing parameter corresponding to `glmnet::alpha` is not stated in the paper text, so `alpha_enet=0.5` is exposed as a configurable implementation choice.


In [ ]:

# Run once if needed:
# %pip install numpy pandas scipy scikit-learn statsmodels

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import LinearSVC
import statsmodels.api as sm


In [ ]:

CONFIG = {
    "p": 20,
    "true_te": 0.5,
    "alpha_enet": 0.5,
    "nfolds": 5,
    "gamma1_elrt": 0.5,
    "gamma1_esvms": 1.0,
    "gamma2": 1.0,
    "coef_tol": 1e-7,
    "treatment_mode": "bernoulli_logit",  # or "paper_literal"
}


## 1. Synthetic data-generating process

In [ ]:

def compound_sigma(p, rho):
    S = np.full((p,p), rho, dtype=float)
    np.fill_diagonal(S, 1.0)
    return S

def scenario_coefficients(scenario, p=20):
    theta = np.zeros(p)
    beta = np.zeros(p)

    if scenario == 1:
        theta[:4] = [.6,.6,.6,.6]
        beta[[0,1,4,5]] = [1,1,1,1]
    elif scenario == 2:
        theta[:4] = [.6,.6,.6,.6]
        beta[[0,1,4,5]] = [.4,.4,1,1]
    elif scenario == 3:
        theta[:4] = [.2,.2,.6,.6]
        beta[[0,1,4,5]] = [1,1,1,1]
    elif scenario == 4:
        theta[:4] = [.6,.6,.6,.6]
        beta[[0,1,4,5]] = [1,1,1.8,1.8]
    else:
        raise ValueError("scenario must be 1..4")
    return theta, beta

def simulate_scenario(n, rho, scenario, seed, treatment_mode=None):
    if treatment_mode is None:
        treatment_mode = CONFIG["treatment_mode"]

    rng = np.random.default_rng(seed)
    p = CONFIG["p"]
    theta, beta = scenario_coefficients(scenario, p)

    X = rng.multivariate_normal(
        np.zeros(p), compound_sigma(p, rho), size=n
    )
    eta = X @ beta

    if treatment_mode == "bernoulli_logit":
        T = rng.binomial(1, expit(eta))
    elif treatment_mode == "paper_literal":
        eps_treat = rng.normal(size=n)
        T = (expit(eta) < eps_treat).astype(int)
    else:
        raise ValueError("unknown treatment_mode")

    Y = CONFIG["true_te"]*T + X@theta + rng.normal(size=n)

    df = pd.DataFrame(X, columns=[f"X{i}" for i in range(1,p+1)])
    df.insert(0, "T", T)
    df.insert(0, "Y", Y)
    return df


## 2. Weighted elastic-net solvers

In [ ]:

def soft_threshold(x,t):
    return np.sign(x)*np.maximum(np.abs(x)-t,0.0)

def gaussian_weighted_enet(X,y,w,lam,alpha,init=None,max_iter=2500,tol=1e-6):
    X=np.asarray(X,float); y=np.asarray(y,float); w=np.asarray(w,float)
    n,p=X.shape
    if init is None:
        b0=y.mean(); b=np.zeros(p)
    else:
        b0,b=init; b=b.copy()

    L=(np.linalg.norm(X,2)**2)/n + lam*(1-alpha)
    L=max(L,1e-8)
    z0=b0; z=b.copy(); tk=1.0

    for _ in range(max_iter):
        r=(z0+X@z)-y
        g0=r.mean()
        g=X.T@r/n + lam*(1-alpha)*z
        b0n=z0-g0/L
        bn=soft_threshold(z-g/L,lam*alpha*w/L)

        delta=max(abs(b0n-b0),np.max(np.abs(bn-b)))
        tnew=(1+np.sqrt(1+4*tk*tk))/2
        mom=(tk-1)/tnew
        z0=b0n+mom*(b0n-b0)
        z=bn+mom*(bn-b)
        b0,b,tk=b0n,bn,tnew
        if delta<tol: break
    return b0,b

def logistic_weighted_enet(X,y,w,lam,alpha,init=None,max_iter=2500,tol=1e-6):
    X=np.asarray(X,float); y=np.asarray(y,float); w=np.asarray(w,float)
    n,p=X.shape
    if init is None:
        mu=np.clip(y.mean(),1e-6,1-1e-6)
        b0=np.log(mu/(1-mu)); b=np.zeros(p)
    else:
        b0,b=init; b=b.copy()

    L=(np.linalg.norm(X,2)**2)/(4*n) + lam*(1-alpha)
    L=max(L,.25)
    z0=b0; z=b.copy(); tk=1.0

    for _ in range(max_iter):
        ph=expit(z0+X@z)
        r=ph-y
        g0=r.mean()
        g=X.T@r/n + lam*(1-alpha)*z
        b0n=z0-g0/L
        bn=soft_threshold(z-g/L,lam*alpha*w/L)

        delta=max(abs(b0n-b0),np.max(np.abs(bn-b)))
        tnew=(1+np.sqrt(1+4*tk*tk))/2
        mom=(tk-1)/tnew
        z0=b0n+mom*(b0n-b0)
        z=bn+mom*(bn-b)
        b0,b,tk=b0n,bn,tnew
        if delta<tol: break
    return b0,b

def cv_weighted_enet(X,y,w,family="gaussian",alpha=.5,nfolds=5,seed=1,nlambda=30):
    X=np.asarray(X,float); y=np.asarray(y,float); w=np.asarray(w,float)
    grid=np.geomspace(1.0,1e-4,nlambda)

    splitter=(StratifiedKFold(n_splits=nfolds,shuffle=True,random_state=seed)
              if family=="binomial"
              else KFold(n_splits=nfolds,shuffle=True,random_state=seed))

    losses=np.zeros((nfolds,nlambda))
    iterator=splitter.split(X,y if family=="binomial" else None)

    for f,(tr,va) in enumerate(iterator):
        init=None
        for j,lam in enumerate(grid):
            if family=="gaussian":
                b0,b=gaussian_weighted_enet(X[tr],y[tr],w,lam,alpha,init)
                pred=b0+X[va]@b
                losses[f,j]=np.mean((y[va]-pred)**2)
            else:
                b0,b=logistic_weighted_enet(X[tr],y[tr],w,lam,alpha,init)
                p=np.clip(expit(b0+X[va]@b),1e-10,1-1e-10)
                losses[f,j]=-2*np.mean(y[va]*np.log(p)+(1-y[va])*np.log(1-p))
            init=(b0,b)

    mean=losses.mean(0)
    se=losses.std(0,ddof=1)/np.sqrt(nfolds)
    jmin=np.argmin(mean)
    cutoff=mean[jmin]+se[jmin]
    j1=np.where(mean<=cutoff)[0][0]
    lam=grid[j1]

    if family=="gaussian":
        b0,b=gaussian_weighted_enet(X,y,w,lam,alpha)
    else:
        b0,b=logistic_weighted_enet(X,y,w,lam,alpha)

    return {"intercept":b0,"coef":b,"lambda_1se":lam}


## 3. Proposed enhanced three-stage framework

In [ ]:

def safe_inverse_power(x,gamma,eps=1e-8,cap=1e12):
    return np.minimum(np.maximum(np.abs(x),eps)**(-gamma),cap)

def proposed_three_stage(df, prior="LR", smoother="tanh",
                         gamma1=None, gamma2=1.0, seed=1):
    vars_=[f"X{i}" for i in range(1,CONFIG["p"]+1)]
    X=df[vars_].to_numpy(float)
    Y=df["Y"].to_numpy(float)
    T=df["T"].to_numpy(int)

    Xs=StandardScaler().fit_transform(X)

    # Stage 1: exposure model supplies prior information
    if prior=="LR":
        m=LogisticRegression(penalty=None,solver="lbfgs",max_iter=5000)
        m.fit(Xs,T)
        beta=m.coef_.ravel()
    elif prior=="SVM":
        m=LinearSVC(C=1.0,dual="auto",max_iter=10000)
        m.fit(Xs,T)
        beta=m.coef_.ravel()
    else:
        raise ValueError

    if gamma1 is None:
        gamma1=(CONFIG["gamma1_elrt"] if smoother=="tanh"
                else CONFIG["gamma1_esvms"])

    if smoother=="tanh":
        w1=np.maximum(np.tanh(np.abs(beta)),1e-8)**gamma1
    elif smoother=="sigmoid":
        w1=(1/(1+np.exp(-np.abs(beta))))**gamma1
    else:
        raise ValueError

    # Stage 2: weighted outcome elastic net
    f2=cv_weighted_enet(
        Xs,Y,w1,"gaussian",CONFIG["alpha_enet"],CONFIG["nfolds"],seed
    )
    theta=f2["coef"]

    # Stage 3: adaptive weights from Stage-2 EN coefficients
    psi=safe_inverse_power(theta,gamma2)

    # Final adaptive elastic net
    f3=cv_weighted_enet(
        Xs,Y,psi,"gaussian",CONFIG["alpha_enet"],CONFIG["nfolds"],seed+10000
    )
    final=f3["coef"]

    selected=[vars_[i] for i in np.flatnonzero(np.abs(final)>CONFIG["coef_tol"])]

    return {
        "selected":selected,
        "beta_prior":beta,
        "penalty_stage2":w1,
        "theta_stage2":theta,
        "penalty_stage3":psi,
        "theta_final":final
    }

def fit_enh_elrt(df,seed=1):
    return proposed_three_stage(
        df,"LR","tanh",CONFIG["gamma1_elrt"],CONFIG["gamma2"],seed
    )

def fit_enh_esvms(df,seed=1):
    return proposed_three_stage(
        df,"SVM","sigmoid",CONFIG["gamma1_esvms"],CONFIG["gamma2"],seed
    )


## 4. OAENet comparator

In [ ]:

def fit_oaenet(df,gamma=3,seed=1):
    vars_=[f"X{i}" for i in range(1,CONFIG["p"]+1)]
    X=df[vars_].to_numpy(float)
    Y=df["Y"].to_numpy(float)
    T=df["T"].to_numpy(int)

    Xs=StandardScaler().fit_transform(X)

    # OAENet Stage 1: outcome OLS
    theta=LinearRegression().fit(Xs,Y).coef_

    # Adaptive weights from OLS coefficients
    w=safe_inverse_power(theta,gamma)

    # OAENet Stage 2: weighted logistic elastic net
    f=cv_weighted_enet(
        Xs,T,w,"binomial",CONFIG["alpha_enet"],CONFIG["nfolds"],seed
    )
    b=f["coef"]
    selected=[vars_[i] for i in np.flatnonzero(np.abs(b)>CONFIG["coef_tol"])]

    return {
        "selected":selected,
        "theta_ols":theta,
        "penalty":w,
        "beta_treatment":b
    }


## 5. One-to-one nearest matching and ATT

In [ ]:

def estimate_att(df, selected):
    if len(selected)==0:
        return np.nan

    X=df[selected].to_numpy(float)
    T=df["T"].to_numpy(int)

    ps=LogisticRegression(penalty=None,solver="lbfgs",max_iter=5000)
    ps.fit(X,T)
    score=ps.predict_proba(X)[:,1]
    eps=1e-8
    p=np.clip(score,eps,1-eps)
    logit=np.log(p/(1-p))

    treated=np.where(T==1)[0]
    controls=set(np.where(T==0)[0].tolist())
    pairs=[]

    for ti in treated[np.argsort(-logit[treated])]:
        if not controls: break
        cc=np.array(sorted(controls))
        ci=int(cc[np.argmin(np.abs(logit[cc]-logit[ti]))])
        pairs.append((ti,ci))
        controls.remove(ci)

    if not pairs:
        return np.nan

    rows=[i for pair in pairs for i in pair]
    md=df.iloc[rows].copy()

    Xreg=sm.add_constant(md[["T"]+selected].astype(float),has_constant="add")
    fit=sm.OLS(md["Y"].astype(float),Xreg).fit()
    return float(fit.params["T"])


## 6. Direct method comparison

In [ ]:

def run_one_comparison(n=500,rho=.25,scenario=1,seed=1,treatment_mode=None):
    df=simulate_scenario(n,rho,scenario,seed,treatment_mode)

    fits={
        "Enh_ELRT":fit_enh_elrt(df,seed),
        "Enh_ESVMS":fit_enh_esvms(df,seed),
        "OAENet":fit_oaenet(df,seed=seed)
    }

    target=["X1","X2","X3","X4"]
    att_target=estimate_att(df,target)

    rows=[]
    for name,fit in fits.items():
        att=estimate_att(df,fit["selected"])
        rows.append({
            "method":name,
            "n":n,"rho":rho,"scenario":scenario,"seed":seed,
            "n_selected":len(fit["selected"]),
            "selected":",".join(fit["selected"]),
            "att":att,
            "target_att":att_target,
            "abs_selection_bias":abs(att-att_target),
            "pct_selection_bias":100*abs(att-att_target)/CONFIG["true_te"]
        })

    rows.append({
        "method":"Target","n":n,"rho":rho,"scenario":scenario,"seed":seed,
        "n_selected":4,"selected":",".join(target),
        "att":att_target,"target_att":att_target,
        "abs_selection_bias":0.0,"pct_selection_bias":0.0
    })

    return pd.DataFrame(rows)

one = run_one_comparison(n=500,rho=.25,scenario=1,seed=1)
one


## 7. Full published grid: 1,440 generated datasets

In [ ]:

def run_full_experiment(
    Ns=(200,500,1000),
    rhos=(0,.25,.5,.75),
    scenarios=(1,2,3,4),
    seeds=range(1,31),
    treatment_mode=None,
    verbose=True
):
    seeds=list(seeds)
    total=len(Ns)*len(rhos)*len(scenarios)*len(seeds)
    out=[]
    k=0

    for scenario in scenarios:
        for n in Ns:
            for rho in rhos:
                for seed in seeds:
                    k+=1
                    if verbose:
                        print(f"[{k}/{total}] s={scenario} n={n} rho={rho} seed={seed}")
                    try:
                        out.append(run_one_comparison(
                            n,rho,scenario,seed,treatment_mode
                        ))
                    except Exception as e:
                        print("FAILED:",e)

    return pd.concat(out,ignore_index=True)

# full = run_full_experiment()
# full.to_csv("three_stage_vs_oaenet_full_results.csv",index=False)


## 8. Summaries and selection frequency

In [ ]:

def summarize_results(results):
    return (
        results.groupby(["method","scenario","n","rho"])
        .agg(
            mean_att=("att","mean"),
            sd_att=("att","std"),
            mean_abs_selection_bias=("abs_selection_bias","mean"),
            mean_pct_selection_bias=("pct_selection_bias","mean"),
            mean_n_selected=("n_selected","mean")
        )
        .reset_index()
    )

def selection_frequency(n=500,rho=.25,scenario=1,seeds=range(1,31),
                        treatment_mode=None):
    seeds=list(seeds)
    vars_=[f"X{i}" for i in range(1,CONFIG["p"]+1)]
    methods=["Enh_ELRT","Enh_ESVMS","OAENet"]
    counts={m:pd.Series(0.0,index=vars_) for m in methods}

    for seed in seeds:
        df=simulate_scenario(n,rho,scenario,seed,treatment_mode)
        fits={
            "Enh_ELRT":fit_enh_elrt(df,seed),
            "Enh_ESVMS":fit_enh_esvms(df,seed),
            "OAENet":fit_oaenet(df,seed=seed)
        }
        for m in methods:
            counts[m].loc[fits[m]["selected"]] += 1

    return pd.DataFrame({
        m:counts[m]/len(seeds) for m in methods
    })

# Example:
# sf = selection_frequency(n=500,rho=.25,scenario=1)
# sf



## Variable roles in the four main scenarios

- `X1, X2`: confounders \(X_C\)
- `X3, X4`: pure outcome predictors \(X_P\)
- `X5, X6`: pure treatment predictors \(X_T\)
- `X7, ..., X20`: noise \(X_S\)

Therefore, the oracle feature set is always `{X1, X2, X3, X4}` in these four scenarios.

The comparison follows the paper's evaluation logic: each selected feature set is used for one-to-one nearest-neighbor matching, followed by regression on the matched data; selection bias is measured relative to the target-model ATT.
